# 13.4 跨模态 (Cross-Modal)

跨模态技术解决不同模态之间的对齐、融合和统一问题，是多模态AI的核心基础设施。

## 跨模态技术层次

| 层次 | 目标 | 方法 | 代表工作 |
|------|------|------|----------|
| 对齐 | 不同模态映射到共享空间 | 对比学习 | CLIP, ALIGN |
| 融合 | 多模态信息合并 | 交叉注意力/门控 | Flamingo, BEiT-3 |
| 统一 | 单一模型处理所有模态 | 统一Tokenizer | GPT-4o, Gemini |
| 生成 | 跨模态内容生成 | 扩散模型/流匹配 | DALL-E 3, Sora |

## 1. 模态对齐（CLIP）

CLIP是跨模态对齐的基础方法，通过对比学习将图像和文本映射到共享空间。

### CLIP训练
- **数据**：4亿图像-文本对
- **损失**：InfoNCE对比损失（双向）
- **温度参数**：可学习的logit_scale

### CLIP的工业应用
- **零样本分类**：不需要微调即可分类
- **检索**：图像-文本跨模态检索
- **VLM视觉编码器**：作为LLaVA等模型的视觉backbone
- **评估**：CLIP Score用于评估生成质量

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

torch.manual_seed(42)

class CLIPModel(nn.Module):
    def __init__(self, d_image=64, d_text=64, d_shared=32, temperature=0.07):
        super().__init__()
        self.image_encoder = nn.Sequential(
            nn.Linear(d_image, 128), nn.ReLU(),
            nn.Linear(128, d_shared)
        )
        self.text_encoder = nn.Sequential(
            nn.Linear(d_text, 128), nn.ReLU(),
            nn.Linear(128, d_shared)
        )
        self.logit_scale = nn.Parameter(torch.ones(1) * math.log(1 / temperature))

    def encode_image(self, img_features):
        return F.normalize(self.image_encoder(img_features), dim=-1)

    def encode_text(self, txt_features):
        return F.normalize(self.text_encoder(txt_features), dim=-1)

    def forward(self, img_features, txt_features):
        img_emb = self.encode_image(img_features)
        txt_emb = self.encode_text(txt_features)
        scale = self.logit_scale.exp().clamp(max=100)
        logits = scale * img_emb @ txt_emb.T
        labels = torch.arange(logits.shape[0], device=logits.device)
        loss_i2t = F.cross_entropy(logits, labels)
        loss_t2i = F.cross_entropy(logits.T, labels)
        loss = (loss_i2t + loss_t2i) / 2
        return loss, logits

    @torch.no_grad()
    def zero_shot_classify(self, img_features, class_embeddings):
        img_emb = self.encode_image(img_features)
        scale = self.logit_scale.exp().clamp(max=100)
        logits = scale * img_emb @ class_embeddings.T
        return logits

clip = CLIPModel()
img_features = torch.randn(8, 64)
txt_features = torch.randn(8, 64)

optimizer = torch.optim.AdamW(clip.parameters(), lr=1e-3)

print('=== CLIP Training ===')
for epoch in range(30):
    loss, sim = clip(img_features, txt_features)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    if (epoch + 1) % 10 == 0:
        with torch.no_grad():
            diag = F.softmax(sim, dim=-1).diag().mean().item()
        print(f'Epoch {epoch+1}: loss={loss.item():.4f}, diag_prob={diag:.3f}')

class_embs = F.normalize(torch.randn(5, 32), dim=-1)
test_img = torch.randn(3, 64)
zs_logits = clip.zero_shot_classify(test_img, class_embs)
print(f'\nZero-shot classification:')
print(f'  Predicted classes: {zs_logits.argmax(dim=-1).tolist()}')
print(f'  Confidence: {zs_logits.softmax(dim=-1).max(dim=-1).values.tolist()}')

print(f'\nKey: CLIP aligns modalities via contrastive learning.')
print(f'Zero-shot classification is a key advantage of CLIP.')

## 2. 模态融合策略

模态融合决定如何将不同模态的信息合并，直接影响多模态模型的效果。

### 融合策略对比
| 策略 | 融合位置 | 参数量 | 效果 | 代表 |
|------|----------|--------|------|------|
| 早期融合 | 输入层 | 少 | 弱 | LLaVA |
| 交叉注意力 | 中间层 | 中 | 强 | Flamingo |
| 门控融合 | 中间层 | 中 | 强 | BEiT-3 |
| 晚期融合 | 输出层 | 少 | 弱 | 简单集成 |

### 注意力掩码策略
在统一模型中，不同模态的token需要不同的注意力掩码：
- **全注意力**：所有token互相看到
- **因果注意力**：文本token只能看到之前的token
- **模态内注意力**：视觉token只看视觉token

In [ ]:
class GatedFusion(nn.Module):
    def __init__(self, d_model=128, n_heads=2):
        super().__init__()
        self.gate = nn.Sequential(
            nn.Linear(d_model * 2, d_model),
            nn.Sigmoid()
        )
        self.cross_attn = nn.MultiheadAttention(d_model, n_heads, batch_first=True)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x_primary, x_secondary):
        attn_out, _ = self.cross_attn(x_primary, x_secondary, x_secondary)
        attn_out = self.norm1(x_primary + attn_out)
        gate_input = torch.cat([x_primary, attn_out], dim=-1)
        gate = self.gate(gate_input)
        fused = gate * attn_out + (1 - gate) * x_primary
        return self.norm2(fused)

class MultiModalAttentionMask:
    def __init__(self, n_vis_tokens=16, n_txt_tokens=10):
        self.n_vis = n_vis_tokens
        self.n_txt = n_txt_tokens

    def full_attention(self):
        total = self.n_vis + self.n_txt
        return torch.ones(total, total)

    def causal_with_vision(self):
        total = self.n_vis + self.n_txt
        mask = torch.zeros(total, total)
        mask[:self.n_vis, :self.n_vis] = 1
        mask[:self.n_vis, self.n_vis:] = 1
        for i in range(self.n_vis, total):
            mask[i, :i+1] = 1
        return mask

x_vis = torch.randn(2, 16, 128)
x_txt = torch.randn(2, 10, 128)

gate_fusion = GatedFusion(d_model=128)
fused = gate_fusion(x_txt, x_vis)

print('=== Modality Fusion ===')
print(f'Text input: {x_txt.shape}')
print(f'Vision input: {x_vis.shape}')
print(f'Gated fusion output: {fused.shape}')

mask_builder = MultiModalAttentionMask(n_vis_tokens=16, n_txt_tokens=10)
causal_mask = mask_builder.causal_with_vision()

print(f'\nCausal attention mask (vision+text):')
print(f'  Vision tokens can see all other tokens')
print(f'  Text tokens follow causal (autoregressive) mask')
print(f'  Mask shape: {causal_mask.shape}')
print(f'  Vision-Vision block: all 1s (bidirectional)')
print(f'  Text-Text block: lower triangular (causal)')

print(f'\nKey: Gated fusion learns when to use cross-modal vs intra-modal info.')
print(f'Causal mask enables autoregressive generation with vision context.')

## 3. 统一多模态模型

统一多模态模型用单一Transformer处理所有模态，是当前最前沿的方向。

### 统一模型的关键设计

1. **统一Tokenizer**
   - 文本：BPE分词
   - 图像：ViT patch → 视觉token
   - 音频：EnCodec → 音频token
   - 所有模态的token在同一个词表中

2. **统一训练目标**
   - 所有模态共享next-token prediction目标
   - 文本：预测下一个文本token
   - 图像：预测下一个视觉token
   - 音频：预测下一个音频token

3. **模态标记**
   - 用特殊token标记不同模态的开始和结束
   - 如 `<image>...</image>`, `<audio>...</audio>`

### 代表模型
- **GPT-4o**：OpenAI的统一多模态模型
- **Gemini**：Google的原生多模态模型
- **Chameleon**：Meta的早期融合多模态模型

In [ ]:
class UnifiedMultiModalModel(nn.Module):
    TEXT = 0
    IMAGE = 1
    AUDIO = 2

    def __init__(self, d_model=128, text_vocab=500, img_vocab=1024, audio_vocab=1024,
                 n_heads=2, n_layers=2):
        super().__init__()
        total_vocab = text_vocab + img_vocab + audio_vocab
        self.embed = nn.Embedding(total_vocab, d_model)
        self.modality_embed = nn.Embedding(3, d_model)
        self.text_vocab = text_vocab
        self.img_vocab = img_vocab
        self.audio_vocab = audio_vocab

        layer = nn.TransformerEncoderLayer(d_model, n_heads, d_model * 4, batch_first=True)
        self.encoder = nn.TransformerEncoder(layer, n_layers)
        self.head = nn.Linear(d_model, total_vocab)

    def tokenize(self, tokens, modality):
        offset = modality * max(self.text_vocab, self.img_vocab, self.audio_vocab)
        return tokens + offset

    def forward(self, tokens, modalities):
        token_emb = self.embed(tokens)
        mod_emb = self.modality_embed(modalities)
        h = self.encoder(token_emb + mod_emb)
        return self.head(h)

    @torch.no_grad()
    def generate(self, tokens, modalities, max_new=10, temperature=1.0):
        for _ in range(max_new):
            logits = self.forward(tokens, modalities)
            next_logits = logits[:, -1, :] / temperature
            probs = F.softmax(next_logits, dim=-1)
            next_token = probs.multinomial(1)
            tokens = torch.cat([tokens, next_token], dim=1)
            modalities = torch.cat([modalities, torch.zeros(1, 1, dtype=torch.long)], dim=1)
        return tokens, modalities

model = UnifiedMultiModalModel()

text_tokens = torch.randint(0, 500, (1, 5))
img_tokens = torch.randint(0, 1024, (1, 8))
all_tokens = torch.cat([text_tokens, img_tokens], dim=1)
all_modalities = torch.tensor([[0,0,0,0,0,1,1,1,1,1,1,1,1]])

logits = model(all_tokens, all_modalities)
gen_tokens, gen_mods = model.generate(all_tokens, all_modalities, max_new=5)

print('=== Unified Multi-Modal Model ===')
print(f'Input: {all_tokens.shape} tokens, {all_modalities.shape} modality labels')
print(f'Output logits: {logits.shape}')
print(f'Generated: {gen_tokens.shape} tokens')
print(f'\nModality breakdown:')
print(f'  Text tokens (mod=0): {(all_modalities == 0).sum().item()}')
print(f'  Image tokens (mod=1): {(all_modalities == 1).sum().item()}')

print(f'\nKey: Unified model processes all modalities with shared Transformer.')
print(f'Modality embeddings distinguish different input types.')
print(f'Single next-token prediction objective across all modalities.')

print(f'\n=== Cross-Modal Industrial Best Practices ===')
print(f'1. CLIP is the foundation: always start with CLIP-aligned encoders')
print(f'2. Projection-based (LLaVA) is the safest starting architecture')
print(f'3. Cross-attention (Flamingo) for deeper fusion when needed')
print(f'4. Unified model (GPT-4o style) is the future but requires massive resources')
print(f'5. Always evaluate cross-modal alignment quality before downstream tasks')
print(f'6. Modality dropout during training improves robustness')